# Bronze ingestion — ARQLMED (SCADA measurement archive)

This notebook takes the raw `working_arqlmed` table (loaded earlier from the source
CSV export), profiles it for data-quality issues, and persists it unchanged into the
**bronze** layer as `hive_metastore.bronze.bronze_arqlmed`.

**Raw columns referenced below** (inferred from usage, not a documented schema):
- `ID` — measurement identifier in the raw archive. Note its cardinality (~39k distinct
  values) is far larger than the ~1,200-transformer fleet seen downstream, so this is a
  point/signal-level identifier rather than the transformer ID.
- `ID_TIME` — time key, stored as a `yyyymmdd`-style integer (observed range below).
- `ID_ARQLMED` — per-record identifier in the archive.

| n_rows | n_ids | min_date | max_date |
|---|---|---|---|
| 896,743,685 | 39,261 | 2023-07-04 | 2024-07-03 |

## 1. Setup

Load shared helpers and Spark imports (`F`, etc.) from the project utilities notebook.

In [0]:
%run /Workspace/Users/daniel.branco@cgi.com/Tese/00_Utils

## 2. Load the raw SCADA archive

Read the source table written by the loader notebook. A 5-row preview and the schema
are printed to confirm the columns and types came through as expected before profiling.

In [0]:
df = spark.read.table("hive_metastore.default.working_arqlmed")
display(df.limit(5))
df.printSchema()

## 3. Data profiling

A quick data-quality sweep over the full table before promoting it to bronze. None of
these cells modify the data — they only summarise it. The goal is to know the row volume,
time coverage, missingness, and how often columns hold a literal zero, so that the
cleaning decisions made later (silver) are grounded in the actual raw distribution.

### 3.1 Volume and time coverage

Row count, distinct `ID` count, and the min/max of `ID_TIME` (the values recorded in the
overview table above).

In [0]:
summary = df.agg(
    F.count("*").alias("n_rows"),
    F.countDistinct("ID").alias("n_ids"),
    F.min("ID_TIME").alias("min_date"),
    F.max("ID_TIME").alias("max_date"),
)
display(summary)

### 3.2 Missing values

Count nulls per column, then express each as a fraction of the total row count `n`.
`nulls` shows absolute counts; `nulls_pct` shows the proportion (0–1) per column.

In [0]:
n = df.count()

nulls = df.agg(*[
    F.sum(F.col(c).isNull().cast("int")).alias(f"null_{c}")
    for c in df.columns
])

nulls_pct = nulls.select(*[
    (F.col(c) / F.lit(n)).alias(c.replace("null_", "pct_null_"))
    for c in nulls.columns
])

display(nulls)
display(nulls_pct)

### 3.3 Zero values

Count, per column, how many rows hold a literal `"0"` after trimming whitespace. Because
the source was read with `inferSchema=false`, every column is a string, so this is an
exact string match against `"0"` — it will **not** catch other zero spellings such as
`"0.0"`, `"0.00"`, or a localised `"0,0"`. Useful as a rough flag for all-zero / inactive
measurement rows, which the silver layer filters out.

In [0]:
agg_exprs = [
    F.sum((F.trim(F.col(c)) == F.lit("0")).cast("int")).alias(c)
    for c in df.columns
]

zero_counts = df.agg(*agg_exprs)
display(zero_counts)

Convert the zero counts to proportions of the total row count, mirroring the null
percentages above.

In [0]:
zero_pct = zero_counts.select(*[
    (F.col(c) / F.lit(n)).alias(c)
    for c in zero_counts.columns
])

display(zero_pct)

## 4. Write to the bronze layer

Persist the raw table, unchanged, as a managed Delta table. `mode("overwrite")` plus
`overwriteSchema=true` makes the cell safely re-runnable: re-executing replaces the table
and its schema rather than erroring on an existing table.

**Target:** `hive_metastore.bronze.bronze_arqlmed`

In [0]:
target_catalog = "hive_metastore"     # change this
target_schema = "bronze"
target_table = "bronze_arqlmed"  # change this

full_name = f"{target_catalog}.{target_schema}.{target_table}"


In [0]:
#spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_catalog}.{target_schema}")

Write to Delta.

In [0]:
(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(full_name)
)

display(spark.table(full_name).limit(20))
